*Notebook for uploading, exploring and cleaning the data.*

In [56]:
import random
import urllib.request
import torch 
import os 

url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
with urllib.request.urlopen(url) as response:
    raw_text = response.read().decode('utf-8')

# Inspecting the corpus 
print(type(raw_text))
print(len(raw_text))
print(raw_text[:2000])

<class 'str'>
551846
LA DIVINA COMMEDIA
di Dante Alighieri
INFERNO



Inferno: Canto I

  Nel mezzo del cammin di nostra vita
mi ritrovai per una selva oscura
ché la diritta via era smarrita.
  Ahi quanto a dir qual era è cosa dura
esta selva selvaggia e aspra e forte
che nel pensier rinova la paura!
  Tant'è amara che poco è più morte;
ma per trattar del ben ch'i' vi trovai,
dirò de l'altre cose ch'i' v'ho scorte.
  Io non so ben ridir com'i' v'intrai,
tant'era pien di sonno a quel punto
che la verace via abbandonai.
  Ma poi ch'i' fui al piè d'un colle giunto,
là dove terminava quella valle
che m'avea di paura il cor compunto,
  guardai in alto, e vidi le sue spalle
vestite già de' raggi del pianeta
che mena dritto altrui per ogne calle.
  Allor fu la paura un poco queta
che nel lago del cor m'era durata
la notte ch'i' passai con tanta pieta.
  E come quei che con lena affannata
uscito fuor del pelago a la riva
si volge a l'acqua perigliosa e guata,
  così l'animo mio, ch'ancor fuggi

### 1 - EDA

In [57]:
# Look for actual line breaks (newlines) and swapping them with a literal backslash and the letter "n".
print(raw_text[:200].replace("\n", "\\n"))

print(repr(raw_text[:100])) #check for string representations

\n  Ahi quanto a dir era smarrita. vita
'LA DIVINA COMMEDIA\r\ndi Dante Alighieri\r\nINFERNO\r\n\r\n\r\n\r\nInferno: Canto I\r\n\r\n  Nel mezzo del cammin di'


In [58]:
lines = raw_text.splitlines()

print(len(lines))
print(lines[:20])

14753
['LA DIVINA COMMEDIA', 'di Dante Alighieri', 'INFERNO', '', '', '', 'Inferno: Canto I', '', '  Nel mezzo del cammin di nostra vita', 'mi ritrovai per una selva oscura', 'ché la diritta via era smarrita.', '  Ahi quanto a dir qual era è cosa dura', 'esta selva selvaggia e aspra e forte', 'che nel pensier rinova la paura!', "  Tant'è amara che poco è più morte;", "ma per trattar del ben ch'i' vi trovai,", "dirò de l'altre cose ch'i' v'ho scorte.", "  Io non so ben ridir com'i' v'intrai,", "tant'era pien di sonno a quel punto", 'che la verace via abbandonai.']


In [59]:
words = raw_text.lower().split()

print(words[:50])
print(len(words))

['la', 'divina', 'commedia', 'di', 'dante', 'alighieri', 'inferno', 'inferno:', 'canto', 'i', 'nel', 'mezzo', 'del', 'cammin', 'di', 'nostra', 'vita', 'mi', 'ritrovai', 'per', 'una', 'selva', 'oscura', 'ché', 'la', 'diritta', 'via', 'era', 'smarrita.', 'ahi', 'quanto', 'a', 'dir', 'qual', 'era', 'è', 'cosa', 'dura', 'esta', 'selva', 'selvaggia', 'e', 'aspra', 'e', 'forte', 'che', 'nel', 'pensier', 'rinova', 'la']
96770


In [60]:
print([word for word in words[:100] if any(c in word for c in ",.;:!?")]) # checking for punctuation

['inferno:', 'smarrita.', 'paura!', 'morte;', 'trovai,', 'scorte.', "v'intrai,", 'abbandonai.']


In [61]:
# Handling punctuation
import string

punctuation = set()

for char in raw_text:
    if char in string.punctuation:
        punctuation.add(char)

print(sorted(punctuation))


['!', '"', "'", '(', ')', ',', '-', '.', ':', ';', '?', '~']


### 2 - Tokenisation

In [62]:
# ALternative way. For learning purposes, I implemented the tokenizer manually.
# import re

# tokens = re.findall(r"[^\s,.!?;:\"()~\-]+|[,.!?;:\"()~\-]", text) #find a word OR find a punctuation symbol (except apostrophe)



In [63]:
punctuation = set('!"(),-.:;?~')

def tokenize(text):
    tokens = []
    current_word = ''
    i = 0
    n = len(text)

    while i < n:
        char = text[i]

        if char.isspace():
            if current_word:
                tokens.append(current_word.lower())
            current_word = ''

        elif char == "'":
            next_char = text[i+1] if i+1 < n else ''
            if next_char.islower():
                current_word += char          # elision, e.g. 'l, ch'io
            else:
                if current_word:
                    tokens.append(current_word.lower())
                current_word = ''              # quote mark, drop it

        elif char in punctuation:
            if current_word:
                tokens.append(current_word.lower())
            current_word = ''
            tokens.append(char)

        else:
            current_word += char

        i += 1

    if current_word:
        tokens.append(current_word.lower())

    return tokens


test = "ch'io vi trovai, nella selva oscura."

print(tokenize(test))

["ch'io", 'vi', 'trovai', ',', 'nella', 'selva', 'oscura', '.']


## 3 - Further cleaning

In [64]:
## Handling '-' as author's writing characteristic
hyphen_words = [w for w in words if '-' in w and w != '-']
print(hyphen_words)



['-.', '-;', '-,', '-,', '-;', '-,', 'differente-']


In [65]:
# This function handles hyphenated words, treating them as a single token.
def dehyphenate(lines):
    new_lines = []
    i = 0

    while i < len(lines):
        line = lines[i]
        if line.endswith('-') and i + 1 < len(lines):
            next_line = lines[i + 1]

            if next_line and next_line[0].islower():
                new_lines.append(line[:-1] + next_line)
                i += 2
                continue
        
        new_lines.append(line)
        i += 1

    return new_lines


lines = dehyphenate(lines)
print([l for l in lines if 'differente' in l])

["qual d'una pianta, in tanto differente,", 'par differente, non da denso e raro;', 'e differentemente han dolce vita', '  così quelle carole, differentemente danzando, de la sua ricchezza']


In [66]:
# Exploring cantos' structure
for line in lines:
    if 'Canto' in line:
        print(repr(line))

'Inferno: Canto I'
'Inferno: Canto II'
'Inferno: Canto III'
'Inferno: Canto IV'
'Inferno: Canto V'
'Inferno: Canto VI'
'Inferno: Canto VII'
'Inferno: Canto VIII'
'Inferno: Canto IX'
'Inferno: Canto X'
'Inferno: Canto XI'
'Inferno: Canto XII'
'Inferno: Canto XIII'
'Inferno: Canto XIV'
'Inferno: Canto XV'
'Inferno: Canto XVI'
'Inferno: Canto XVII'
'Inferno: Canto XVIII'
'Inferno: Canto XIX'
'Inferno: Canto XX'
'Inferno: Canto XXI'
'Inferno: Canto XXII'
'Inferno: Canto XXIII'
'Inferno: Canto XXIV'
'Inferno: Canto XXV'
'Inferno: Canto XXVI'
'Inferno: Canto XXVII'
'Inferno: Canto XXVIII'
'Inferno: Canto XXIX'
'Inferno: Canto XXX'
'Inferno: Canto XXXI'
'Inferno: Canto XXXII'
'Inferno: Canto XXXIII'
'Inferno: Canto XXXIV'
'Purgatorio: Canto I'
'Purgatorio: Canto II'
'Purgatorio: Canto III'
'Purgatorio: Canto IV'
'Purgatorio: Canto V'
'Purgatorio: Canto VI'
'Purgatorio: Canto VII'
'Purgatorio: Canto VIII'
'Purgatorio: Canto IX'
'Purgatorio: Canto X'
'Purgatorio: Canto XI'
'Purgatorio: Canto XI

In [67]:
# Canto segmentation for data splitting 

cantica = ('Inferno', 'Purgatorio','Paradiso')
cantos = []
current_canto = None


for line in lines:
    if line.startswith(cantica):

        if current_canto:
            cantos.append(current_canto)

        current_canto = {'name': line.strip(), 'lines': []}
    
    elif current_canto is not None and line.strip():
        current_canto['lines'].append(line)

if current_canto is not None:
    cantos.append(current_canto)

print(len(cantos))
print(cantos[0]["name"])
print(cantos[0]["lines"][:5])

100
Inferno: Canto I
['  Nel mezzo del cammin di nostra vita', 'mi ritrovai per una selva oscura', 'ché la diritta via era smarrita.', '  Ahi quanto a dir qual era è cosa dura', 'esta selva selvaggia e aspra e forte']


## 4 - train/val/test splitting

In [68]:
#Splitting cantos
import random

def split_cantos(cantos, seed= 42, train_set = 0.8, val_set = 0.1):
    random.seed(seed)

    # group cantos by cantica (Inferno / Purgatorio / Paradiso)
    by_cantica={}


    for canto in cantos:
        cantica = canto['name'].split(':')[0]
        by_cantica.setdefault(cantica, []).append(canto)

    train, val, test = [], [], []

    for cantica, group in by_cantica.items():
        random.shuffle(group)
        n = len(group)
        n_train = int(n * train_set)
        n_valid = int(n * val_set)

        train += group[:n_train]
        val += group[n_train: n_train + n_valid]
        test += group[n_train + n_valid:]
    
    return train, val, test


train_cantos, val_cantos, test_cantos = split_cantos(cantos)

print(len(train_cantos), len(val_cantos), len(test_cantos))

79 9 12


## 5 - tokenize per split

In [69]:
def prepare_tokens(cantos):
    tokens = []

    for canto in cantos:
        text = " ".join(canto["lines"])
        tokens.extend(tokenize(text))

    return tokens

train_tokens = prepare_tokens(train_cantos)
val_tokens = prepare_tokens(val_cantos)
test_tokens = prepare_tokens(test_cantos)

print(len(train_tokens))
print(len(val_tokens))
print(len(test_tokens))

print(train_tokens[:10])

89605
10292
13623
['al', 'tornar', 'de', 'la', 'mente', ',', 'che', 'si', 'chiuse', 'dinanzi']


## 6 - Vocabulary and encoding

In [70]:
def build_vocab(tokens):

    unique_tokens = sorted(set(tokens))

    stoi = {
        '<PAD>': 0,
        '<UNK>': 1
    }

    for token in unique_tokens:
        if token not in stoi:
            stoi[token] = len(stoi)

    itos = {index: token for token, index in stoi.items()}

    return stoi, itos


stoi, itos = build_vocab(train_tokens)

print(len(stoi))
print(stoi)

12002
{'<PAD>': 0, '<UNK>': 1, '!': 2, '"': 3, "'al": 4, "'altro": 5, "'consorte": 6, "'divieto": 7, "'este": 8, "'issa": 9, "'l": 10, "'magino": 11, "'mbedue": 12, "'mbestiate": 13, "'mbocche": 14, "'mmonito": 15, "'mo": 16, "'mpacciata": 17, "'mpacciati": 18, "'mpaluda": 19, "'mpaniati": 20, "'mparadisa": 21, "'mpedisce": 22, "'mpediva": 23, "'mpegolate": 24, "'mperché": 25, "'mperio": 26, "'mpetro": 27, "'mprenta": 28, "'mprenti": 29, "'mpresa": 30, "'mpresso": 31, "'n": 32, "'ncarcati": 33, "'ncarco": 34, "'ncendio": 35, "'ncensi": 36, "'nchiese": 37, "'nchiude": 38, "'ncise": 39, "'ncontra": 40, "'ncontro": 41, "'ndarno": 42, "'ndivine": 43, "'nduce": 44, "'nfamia": 45, "'nferno": 46, "'nfiammati": 47, "'nfiata": 48, "'nfin": 49, "'nfino": 50, "'nforco": 51, "'nfuria": 52, "'nganno": 53, "'ngegni": 54, "'ngegno": 55, "'nghiottiva": 56, "'ngrossa": 57, "'nnamora": 58, "'nnanzi": 59, "'nnocenti": 60, "'noi": 61, "'nostro": 62, "'nsegna": 63, "'nsegnerà": 64, "'nsegni": 65, "'nsembre

In [71]:
def encode(tokens, stoi):
    return [stoi.get(token, stoi["<UNK>"]) for token in tokens]

train_ids = encode(train_tokens, stoi)
val_ids = encode(val_tokens, stoi)
test_ids = encode(test_tokens, stoi)


print(sum(1 for id in val_ids if id == 1))  # how many <UNK> in val — should be small but nonzero

851


## 7 - Build dataset

In [72]:
# Dataset construction

def build_dataset(tokens, stoi, block_size):
    X, Y = [], []

    context = [stoi['<PAD>']] * block_size

    for token in tokens:

        target = stoi.get(token, stoi['<UNK>'])
        X.append(context)
        Y.append(target)
    
        context = context[1:] + [target] # sliding context window

    return X, Y


block_size = 5  # treat as a hyperparameter, try 3 / 5 / 8 later

X_train, Y_train = build_dataset(train_tokens, stoi, block_size)
X_val,   Y_val   = build_dataset(val_tokens,   stoi, block_size)
X_test,  Y_test  = build_dataset(test_tokens,  stoi, block_size)

X_train, Y_train = torch.tensor(X_train), torch.tensor(Y_train)
X_val,   Y_val   = torch.tensor(X_val),   torch.tensor(Y_val)
X_test,  Y_test  = torch.tensor(X_test),  torch.tensor(Y_test)

print(X_train.shape, Y_train.shape)
print(X_val.shape,   Y_val.shape)
print(X_test.shape,  Y_test.shape)


torch.Size([89605, 5]) torch.Size([89605])
torch.Size([10292, 5]) torch.Size([10292])
torch.Size([13623, 5]) torch.Size([13623])


## 8 - Saving data

In [76]:
os.makedirs('./data', exist_ok=True)

dante_data = {
    'cantos': cantos,
    'train_tokens': train_tokens, 'val_tokens': val_tokens, 'test_tokens': test_tokens,
    'stoi': stoi, 'itos': itos,
    'block_size': block_size,
    'X_train': X_train, 'Y_train': Y_train,
    'X_val': X_val,     'Y_val': Y_val,
    'X_test': X_test,   'Y_test': Y_test,
}


torch.save(dante_data, './data/dante_preprocessed.pt')